# Data Analysis Demo

A complete data analysis workflow with sample sales data.

**What we'll cover:**
- Loading and exploring data
- Data cleaning
- Statistical analysis
- Visualization
- Using Claude Code for insights

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

print('Libraries loaded!')

## 1. Create Sample Data

We'll create a realistic sales dataset for a fictional company.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate sample sales data
n_records = 500

# Date range: full year 2024
dates = pd.date_range('2024-01-01', '2024-12-31', periods=n_records)

# Products and regions
products = ['Widget A', 'Widget B', 'Gadget X', 'Gadget Y', 'Service Plan']
regions = ['North', 'South', 'East', 'West']
sales_reps = ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve']

# Create DataFrame
df = pd.DataFrame({
    'date': dates,
    'product': np.random.choice(products, n_records),
    'region': np.random.choice(regions, n_records),
    'sales_rep': np.random.choice(sales_reps, n_records),
    'quantity': np.random.randint(1, 50, n_records),
    'unit_price': np.random.uniform(10, 500, n_records).round(2),
    'discount_pct': np.random.choice([0, 5, 10, 15, 20], n_records),
})

# Calculate revenue
df['revenue'] = (df['quantity'] * df['unit_price'] * (1 - df['discount_pct']/100)).round(2)

# Add some nulls for realistic data cleaning practice
df.loc[np.random.choice(df.index, 10), 'discount_pct'] = np.nan
df.loc[np.random.choice(df.index, 5), 'region'] = np.nan

# Extract time features
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['day_of_week'] = df['date'].dt.day_name()

print(f'Created dataset with {len(df)} records')
df.head(10)

## 2. Data Exploration

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(f'Shape: {df.shape}')
print(f'\nColumn types:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Categorical breakdowns
print('=== Sales by Product ===')
print(df['product'].value_counts())
print('\n=== Sales by Region ===')
print(df['region'].value_counts())

## 3. Data Cleaning

In [ ]:
# Create a clean copy
df_clean = df.copy()

# Fill missing discounts with 0 (no discount)
df_clean['discount_pct'] = df_clean['discount_pct'].fillna(0)

# Fill missing regions with 'Unknown'
df_clean['region'] = df_clean['region'].fillna('Unknown')

# Recalculate revenue for filled discounts
df_clean['revenue'] = (df_clean['quantity'] * df_clean['unit_price'] * (1 - df_clean['discount_pct']/100)).round(2)

print('Missing values after cleaning:')
print(df_clean.isnull().sum())

## 4. Analysis & Aggregations

In [ ]:
# Revenue by product
revenue_by_product = df_clean.groupby('product').agg({
    'revenue': ['sum', 'mean', 'count'],
    'quantity': 'sum'
}).round(2)
revenue_by_product.columns = ['total_revenue', 'avg_revenue', 'num_sales', 'total_quantity']
revenue_by_product = revenue_by_product.sort_values('total_revenue', ascending=False)
print('=== Revenue by Product ===')
revenue_by_product

In [ ]:
# Revenue by region
revenue_by_region = df_clean.groupby('region')['revenue'].agg(['sum', 'mean', 'count']).round(2)
revenue_by_region.columns = ['total_revenue', 'avg_revenue', 'num_sales']
revenue_by_region = revenue_by_region.sort_values('total_revenue', ascending=False)
print('=== Revenue by Region ===')
revenue_by_region

In [ ]:
# Monthly trends
monthly = df_clean.groupby('month').agg({
    'revenue': 'sum',
    'quantity': 'sum',
    'date': 'count'
}).rename(columns={'date': 'num_transactions'})
print('=== Monthly Performance ===')
monthly

In [ ]:
# Top sales reps
rep_performance = df_clean.groupby('sales_rep').agg({
    'revenue': ['sum', 'mean'],
    'quantity': 'sum',
    'date': 'count'
}).round(2)
rep_performance.columns = ['total_revenue', 'avg_deal_size', 'total_units', 'num_deals']
rep_performance = rep_performance.sort_values('total_revenue', ascending=False)
print('=== Sales Rep Performance ===')
rep_performance

## 5. Visualizations

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Revenue by Product (bar chart)
revenue_by_product['total_revenue'].plot(kind='barh', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Total Revenue by Product', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Revenue ($)')

# 2. Revenue by Region (pie chart)
revenue_by_region['total_revenue'].plot(kind='pie', ax=axes[0, 1], autopct='%1.1f%%')
axes[0, 1].set_title('Revenue Distribution by Region', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('')

# 3. Monthly Revenue Trend (line chart)
axes[1, 0].plot(monthly.index, monthly['revenue'], marker='o', linewidth=2, markersize=8)
axes[1, 0].fill_between(monthly.index, monthly['revenue'], alpha=0.3)
axes[1, 0].set_title('Monthly Revenue Trend', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Revenue ($)')
axes[1, 0].set_xticks(range(1, 13))

# 4. Sales Rep Performance (bar chart)
rep_performance['total_revenue'].plot(kind='bar', ax=axes[1, 1], color='seagreen')
axes[1, 1].set_title('Revenue by Sales Rep', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Sales Rep')
axes[1, 1].set_ylabel('Revenue ($)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Revenue distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Revenue distribution
axes[0].hist(df_clean['revenue'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(df_clean['revenue'].mean(), color='red', linestyle='--', label=f'Mean: ${df_clean["revenue"].mean():.2f}')
axes[0].axvline(df_clean['revenue'].median(), color='green', linestyle='--', label=f'Median: ${df_clean["revenue"].median():.2f}')
axes[0].set_title('Revenue Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Revenue ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Quantity vs Revenue scatter
axes[1].scatter(df_clean['quantity'], df_clean['revenue'], alpha=0.5, c=df_clean['discount_pct'], cmap='coolwarm')
axes[1].set_title('Quantity vs Revenue (colored by discount)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Quantity')
axes[1].set_ylabel('Revenue ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: Product x Region revenue
pivot = df_clean.pivot_table(values='revenue', index='product', columns='region', aggfunc='sum')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu', linewidths=0.5)
plt.title('Revenue Heatmap: Product vs Region', fontsize=14, fontweight='bold')
plt.show()

## 6. Key Insights

In [ ]:
# Calculate key metrics
total_revenue = df_clean['revenue'].sum()
avg_deal_size = df_clean['revenue'].mean()
total_units = df_clean['quantity'].sum()
top_product = revenue_by_product['total_revenue'].idxmax()
top_region = revenue_by_region['total_revenue'].idxmax()
top_rep = rep_performance['total_revenue'].idxmax()
best_month = monthly['revenue'].idxmax()

print('=' * 50)
print('KEY BUSINESS INSIGHTS')
print('=' * 50)
print(f'\nTotal Revenue: ${total_revenue:,.2f}')
print(f'Average Deal Size: ${avg_deal_size:,.2f}')
print(f'Total Units Sold: {total_units:,}')
print(f'\nTop Product: {top_product}')
print(f'Top Region: {top_region}')
print(f'Top Sales Rep: {top_rep}')
print(f'Best Month: Month {best_month}')
print('=' * 50)

## 7. Ask Claude Code for Analysis

Use Claude Code to get AI-powered insights on the data.

In [ ]:
# Prepare a summary for Claude
summary = f"""
Sales Data Summary:
- Total Revenue: ${total_revenue:,.2f}
- Total Transactions: {len(df_clean)}
- Products: {', '.join(products)}
- Regions: {', '.join(regions)}

Revenue by Product:
{revenue_by_product['total_revenue'].to_string()}

Revenue by Region:
{revenue_by_region['total_revenue'].to_string()}

Monthly Revenue Trend:
{monthly['revenue'].to_string()}
"""
print(summary)

In [ ]:
# Ask Claude for insights (uncomment to run)
# !echo "{summary}" | claude -p "Based on this sales data, provide 3 actionable business recommendations. Be concise."

## 8. Export Results

In [ ]:
# Save cleaned data
# df_clean.to_csv('sales_data_clean.csv', index=False)
# print('Data exported to sales_data_clean.csv')

# Save summary stats
# revenue_by_product.to_csv('revenue_by_product.csv')
# print('Summary exported!')

print('Uncomment above lines to export data')

---

## Summary

This notebook demonstrated:
- Creating and exploring sample data
- Data cleaning (handling missing values)
- Aggregations and groupby operations
- Multiple visualization types
- Extracting key business insights
- Integration with Claude Code for AI analysis

**Next steps:**
- Try modifying the analysis for your own data
- Add more visualizations
- Use Claude Code to generate additional analysis code